#Notebook 00 - Verificador de la carpeta de cuadernos

Comprueba, antes de que empieces a ejecutar, que la carpeta tenga lo que debe tener.

## Qué significa cada estado

| Estado | Qué quiere decir | ¿Bloquea? |
|---|---|---|
| `OK` | El código coincide con el entregado | — |
| `CAMBIADO` | **El código** es distinto del entregado | Solo si no fuiste tú |
| `FALTA` | El archivo no está en la carpeta | Sí, salvo los marcados como opcionales |
| `SOBRA` | Quedó una versión anterior de un cuaderno | Sí: Colab puede abrir el que no es |
| `AJENO` | Hay un `.ipynb` que no pertenece a la cadena | No, pero conviene mirarlo |

## Ejecutar un cuaderno NO lo marca como cambiado

La huella se calcula **solo sobre el texto de las celdas**. Las salidas que Colab guarda dentro
del `.ipynb` al ejecutarlo, los números de ejecución y los metadatos quedan fuera a propósito.

Así, `CAMBIADO` significa de verdad que alguien editó el código, que es la única situación en
la que quieres enterarte. Los cuadernos que ya ejecutaste aparecen marcados como tales, pero
salen `OK`.

## Cómo usarlo

Ejecuta las celdas de arriba abajo. La carpeta se localiza sola buscando
`04_preprocesamiento.ipynb` en tu Drive; si prefieres fijarla a mano, escribe la ruta
en la variable `CARPETA` de la primera celda.
## Numeracion

Los cuadernos se renumeraron en septiembre de 2026 para que el numero del nombre
coincida con el orden de ejecucion: se ejecutan de `01` a `11`, en ese orden.
Si en la carpeta quedan archivos con los nombres viejos, esta comprobacion los marca
como `SOBRA`: muevelos a una subcarpeta para no abrir el que no es.


In [ ]:
# 1. Montar el Drive y localizar la carpeta de los cuadernos
#
# PARCHE: antes se tomaba la PRIMERA coincidencia del Drive, y como hay una copia
# del proyecto del semestre anterior, a veces se verificaba la carpeta equivocada.
# Ahora la carpeta se decide igual que en los once cuadernos: por CARPETA_PROYECTO.
import hashlib
import json
import os
import subprocess

CARPETA_PROYECTO = 'Proyecto Aplicado III'    # <-- la misma linea de los cuadernos
EXCLUIR_RUTAS    = ['/anterior/', '/.Trash', '/.shortcut-targets-by-id/',
                    '/copia de ', '/copy of ']

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except Exception:
    EN_COLAB = False
    print('No se detecto Colab; se verificara la carpeta actual.')

# Si prefieres fijar la ruta a mano, escribela aqui y se usara tal cual.
CARPETA = None      # por ejemplo: '/content/drive/MyDrive/.../Codigos Trabajo de Grado'


def _find(patron, tipo='f', raiz='/content/drive/MyDrive', timeout=300):
    cmd = ['find', raiz, '-type', tipo, '-name', patron]
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception:
        return []
    return [r for r in res.stdout.strip().split('\n') if r]


def _excluida(ruta):
    r = ruta.lower()
    return any(x.lower() in r for x in EXCLUIR_RUTAS)


if CARPETA is None:
    if EN_COLAB:
        todas  = [os.path.dirname(h)
                  for h in _find('04_preprocesamiento.ipynb')]
        validas = [c for c in todas if CARPETA_PROYECTO in c and not _excluida(c)]
        if len(validas) == 1:
            CARPETA = validas[0]
            print('Carpeta localizada.')
            for c in todas:
                if c not in validas:
                    print(f'  (se ignora otra copia: {c})')
        else:
            print()
            print('=' * 78)
            print('NO SE PUDO DECIDIR QUE CARPETA VERIFICAR')
            print('=' * 78)
            for c in todas:
                print(f'    [{"SIRVE     " if c in validas else "descartada"}] {c}')
            if not todas:
                print('    (no se encontro ningun 04_preprocesamiento.ipynb)')
            print()
            print('  Escribe la ruta a mano arriba, en la variable CARPETA,')
            print(f'  o ajusta CARPETA_PROYECTO (ahora: "{CARPETA_PROYECTO}").')
            print('=' * 78)
            raise FileNotFoundError('No hay exactamente una carpeta candidata.')
    else:
        CARPETA = '.'

print('CARPETA :', CARPETA)


## 2. Huellas esperadas y funciones de verificación

In [ ]:
ESPERADO = {
    '01_descarga_nasa_power.ipynb': '0e4a641cc7c09536',
    '02_eda_estacion_cesmag.ipynb': '6e5c37acf14ca98a',
    '03_analisis_espacial_grillas_nasa.ipynb': 'd1887285846e7c3a',
    '04_preprocesamiento.ipynb': '7096de53e8619803',
    '05_serie_diurna_y_validacion_imputacion.ipynb': 'b61d5614c21ea89b',
    '06_eda_post_preprocesamiento.ipynb': 'b7850e5b69d76201',
    '07_feature_engineering.ipynb': '56c50261ddd2e2d7',
    '08_asociacion_particion_ventanas.ipynb': '03d78cb563204e76',
    '09_informer_hiperparametros.ipynb': 'b7a6863edfd26981',
    '10_entrenamiento_variantes_y_evaluacion.ipynb': 'e8d7109a02770de1',
    '11_diagnostico_modelo.ipynb': 'b5a65b56948f4cce',
    'estilo_tg.py': '6e2050ebc69115d0',
}

# Cuadernos que no son imprescindibles para ejecutar la cadena.
OPCIONALES = {
    'estilo_tg.py': 'no bloquea: solo se usa en la Fase 7 para revisar figuras',
}

# Versiones previas de los cuadernos, que ya no deben estar en esta carpeta.
SUPERADOS = [
    # Nombres de la numeracion anterior (renombrados en septiembre de 2026).
    '00_descarga_nasa_power_REVISADO.ipynb',
    '01_eda_estacion_cesmag_REVISADO.ipynb',
    '02_preprocesamiento_REVISADO.ipynb',
    '03_eda_post_preprocesamiento_REVISADO.ipynb',
    '04_feature_engineering_REVISADO.ipynb',
    '05_modelo_informer_entrenamiento_REVISADO.ipynb',
    '06_evaluacion_comparacion_variantes_REVISADO.ipynb',
    '07_diagnostico_modelo_REVISADO.ipynb',
    '08_analisis_espacial_grillas_nasa.ipynb',
    '09_analisis_series_diurno_e_imputacion.ipynb',
    '10_asociacion_particion_ventanas.ipynb',
    'verificar_carpeta.ipynb',
    # Versiones previas a la revision de la entrega 1.
    '00_descarga_nasa_power_v2.ipynb',
    '01_eda_estacion_cesmag.ipynb',
    '02_preprocesamiento.ipynb',
    '03_eda_post_preprocesamiento.ipynb',
    '04_feature_engineering.ipynb',
    '05_modelo_informer_entrenamiento.ipynb',
    '06_evaluacion_comparacion_variantes.ipynb',
    '07_diagnostico_modelo.ipynb',
]

# Archivos que pueden convivir en la carpeta sin pertenecer a la cadena.
TOLERADOS = [
    '00_verificacion_entorno.ipynb',
    '00_verificacion_entorno.py',
    'verificar_carpeta.py',
    'utilidad_diagnostico_rutas.ipynb',
    '00_diagnostico_rutas.ipynb',
]


def huella(ruta):
    """Huella del CONTENIDO REAL del archivo.

    Para un cuaderno se toma solo el texto de las celdas. Las salidas, los
    numeros de ejecucion y los metadatos quedan fuera a proposito: ejecutar un
    cuaderno en Colab los cambia, y eso no significa que el codigo sea otro.
    Asi, CAMBIADO quiere decir de verdad que el codigo es distinto.
    """
    if ruta.endswith('.ipynb'):
        try:
            with open(ruta, 'r', encoding='utf-8') as f:
                nb = json.load(f)
        except Exception:
            return 'ilegible'
        piezas = []
        for c in nb.get('cells', []):
            src = c.get('source', '')
            if isinstance(src, list):
                src = ''.join(src)
            piezas.append(c.get('cell_type', '') + '\x00' + src.replace('\r\n', '\n').rstrip('\n'))
        material = '\x01'.join(piezas).encode('utf-8')
    else:
        with open(ruta, 'rb') as f:
            material = f.read().replace(b'\r\n', b'\n')
    return hashlib.sha256(material).hexdigest()[:16]


def fue_ejecutado(ruta):
    """True si el cuaderno trae salidas guardadas, es decir, si ya se corrio."""
    if not ruta.endswith('.ipynb'):
        return None
    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            nb = json.load(f)
    except Exception:
        return None
    return any(c.get('outputs') for c in nb.get('cells', []))


def verificar(carpeta, mostrar_ejecutados=True):
    print('Verificando:', os.path.abspath(carpeta))
    print('=' * 78)
    faltan, alterados, correctos = [], [], []

    for nombre, esperada in ESPERADO.items():
        ruta = os.path.join(carpeta, nombre)
        if not os.path.exists(ruta):
            faltan.append(nombre)
            continue
        actual = huella(ruta)
        (correctos if actual == esperada else alterados).append((nombre, actual))

    for nombre, _ in correctos:
        marca = ''
        if mostrar_ejecutados and fue_ejecutado(os.path.join(carpeta, nombre)):
            marca = '   (ya ejecutado — las salidas no afectan a la huella)'
        print(f'  OK        {nombre}{marca}')
    for nombre, actual in alterados:
        print(f'  CAMBIADO  {nombre}')
        print(f'            codigo distinto del entregado '
              f'(huella {actual}, se esperaba {ESPERADO[nombre]})')
    for nombre in faltan:
        nota = OPCIONALES.get(nombre, '')
        print(f'  FALTA     {nombre}' + (f'   <- {nota}' if nota else ''))

    viejos = [n for n in SUPERADOS if os.path.exists(os.path.join(carpeta, n))]
    for n in viejos:
        print(f'  SOBRA     {n}   <- version anterior, sacala de esta carpeta')

    try:
        presentes = sorted(os.listdir(carpeta))
    except OSError:
        presentes = []
    extras = [f for f in presentes
              if f.endswith('.ipynb') and f not in ESPERADO
              and f not in SUPERADOS and f not in TOLERADOS]
    for f in extras:
        print(f'  AJENO     {f}   <- no pertenece a la cadena; revisa si debe estar aqui')

    faltan_criticos = [n for n in faltan if n not in OPCIONALES]
    print('=' * 78)
    print(f'  {len(correctos)} de {len(ESPERADO)} archivos con el codigo esperado')
    if alterados:
        print(f'  {len(alterados)} con el codigo cambiado')
        print('     Si los editaste a proposito, esta bien. Si no, vuelve a bajarlos.')
        print('     Ojo: haberlos EJECUTADO no los marca como cambiados; solo editarlos.')
    if faltan:
        print(f'  {len(faltan)} faltan  ({len(faltan_criticos)} de ellos hacen falta para ejecutar)')
    if viejos:
        print(f'  {len(viejos)} versiones anteriores que hay que sacar de la carpeta')

    listo = not faltan_criticos and not viejos
    print()
    if listo and not faltan and not alterados:
        print('  RESULTADO: carpeta lista.')
    elif listo:
        print('  RESULTADO: puedes ejecutar. Lee los avisos de arriba, ninguno bloquea.')
    else:
        print('  RESULTADO: falta algo imprescindible. Revisa las lineas de arriba.')
    return listo


def revisar_figuras(carpeta_base):
    """Comprueba que cada figura exista en los tres formatos."""
    import glob
    sets = {}
    for sub, ext in [('figuras', 'png'), ('figuras_SVG', 'svg'), ('figuras_PDF', 'pdf')]:
        d = os.path.join(carpeta_base, sub)
        sets[ext] = {os.path.splitext(os.path.basename(x))[0]
                     for x in glob.glob(os.path.join(d, f'*.{ext}'))}
        print(f'  {sub:<14} {len(sets[ext]):>3} archivos .{ext}')
    todos = sets['png'] | sets['svg'] | sets['pdf']
    if not todos:
        print()
        print('  Todavia no hay figuras. Es normal si aun no has ejecutado ningun cuaderno.')
        return True
    faltan = [n for n in sorted(todos)
              if not all(n in sets[e] for e in ('png', 'svg', 'pdf'))]
    print()
    print(f'  Figuras distintas: {len(todos)} | completas en los tres formatos: '
          f'{len(todos) - len(faltan)}')
    if faltan:
        print('  INCOMPLETAS (vuelve a ejecutar la celda que las genera):')
        for n in faltan:
            estado = ', '.join(e for e in ('png', 'svg', 'pdf') if n in sets[e])
            print(f'    {n}  ->  solo {estado}')
        return False
    print('  Todas las figuras estan en PNG, SVG y PDF.')
    return True


print('Funciones listas. La huella se calcula sobre el codigo, no sobre las salidas.')

## 3. Resultado de la carpeta

In [ ]:
listo = verificar(CARPETA)

## 4. Las figuras, en los tres formatos

Esta comprobación tiene sentido **después** de ejecutar los cuadernos, no antes. Las tres
carpetas deben tener el mismo número de archivos: si `figuras_PDF/` tiene menos que `figuras/`,
alguna celda falló a mitad y hay que volver a ejecutarla.

In [ ]:
# La carpeta base es la que contiene figuras/, figuras_SVG/ y figuras_PDF/
BASE = CARPETA
print('FIGURAS EN LOS TRES FORMATOS')
print('=' * 78)
figuras_ok = revisar_figuras(BASE)

## 5. Inventario de la carpeta

In [ ]:
import os
print('CONTENIDO COMPLETO DE LA CARPETA')
print('=' * 78)
for f in sorted(os.listdir(CARPETA)):
    ruta = os.path.join(CARPETA, f)
    if os.path.isdir(ruta):
        n = len(os.listdir(ruta))
        print(f'  [carpeta] {f}   ({n} elementos)')
    else:
        print(f'  {os.path.getsize(ruta)/1024:>9.1f} KB  {f}')

## 6. ¿Se está escribiendo en la carpeta correcta?

Esta celda busca en todo el Drive carpetas `figuras`, `figuras_PDF`, `figuras_SVG`,
`datos`, `modelos` y `resultados` que estén **fuera** de la carpeta del proyecto.
Si aparecen, es que algún cuaderno escribió en la copia vieja.


In [ ]:
# 6. Detectar salidas escritas fuera de la carpeta del proyecto
SALIDAS = ['figuras', 'figuras_SVG', 'figuras_PDF', 'datos', 'modelos', 'resultados']

if not EN_COLAB:
    print('Fuera de Colab no hay Drive que revisar.')
else:
    fuera = []
    for nombre in SALIDAS:
        for d in _find(nombre, tipo='d'):
            if os.path.dirname(d) != CARPETA and not d.startswith(CARPETA):
                n = len(os.listdir(d))
                fuera.append((d, n))
    print('SALIDAS FUERA DE LA CARPETA DEL PROYECTO')
    print('=' * 78)
    if not fuera:
        print('  Ninguna. Todo lo que generan los cuadernos esta en:')
        print(f'  {CARPETA}')
    else:
        for d, n in sorted(fuera):
            print(f'  {n:>4} archivos  {d}')
        print()
        print('  Estas carpetas son de una copia vieja del proyecto. No las borro yo:')
        print('  revisa si guardan algo que quieras conservar y, cuando estes seguro,')
        print('  borralas desde Drive para no volver a confundirte.')
    print('=' * 78)
